# 01 - Data prep
Builds a tract-level base table for Miami-Dade and Broward:
FEMA flood-zone share, NFIP claims rate, and SVI. Raw downloads are cached in `data/raw/` (git-ignored).

**Manual inputs needed first** (see `data/README.md`): `data/raw/svi_florida.csv` and `data/raw/housing_units.csv`.

In [ ]:
import sys; sys.path.insert(0, "..")
import geopandas as gpd, pandas as pd
from src import config
from src.data_access import get_tracts, fetch_nfhl, fetch_nfip_claims

## Census tracts

In [ ]:
tracts = get_tracts()
print(len(tracts), "tracts")
bbox = tuple(tracts.total_bounds)
tracts.plot(figsize=(5,6), linewidth=0.2, edgecolor="k");

## FEMA flood hazard zones
Cached to `data/raw/nfhl.gpkg` after the first run. The query can take a few minutes.

In [ ]:
nfhl_path = config.RAW / "nfhl.gpkg"
if nfhl_path.exists():
    nfhl = gpd.read_file(nfhl_path)
else:
    nfhl = fetch_nfhl(bbox)
    nfhl.to_file(nfhl_path, driver="GPKG")
print(len(nfhl), "flood-zone polygons")
nfhl["FLD_ZONE"].value_counts().head(10)

### Share of each tract inside a Special Flood Hazard Area (SFHA)
Area math is done in a projected CRS (`EPSG:2236`), not lat/lon.

In [ ]:
t = tracts.to_crs(config.AREA_CRS)
sfha = nfhl[nfhl["SFHA_TF"] == "T"].to_crs(config.AREA_CRS)
sfha = sfha.dissolve().explode(index_parts=False)[["geometry"]]
sfha["geometry"] = sfha.make_valid()

inter = gpd.overlay(t, sfha, how="intersection", keep_geom_type=True)
inter["a"] = inter.area
t["tract_area"] = t.area
t = t.merge(inter.groupby("GEOID")["a"].sum().rename("sfha_area"), on="GEOID", how="left")
t["sfha_share"] = (t["sfha_area"].fillna(0) / t["tract_area"]).clip(0, 1)
t[["GEOID", "sfha_share"]].describe()

## NFIP claims, SVI, housing units

In [ ]:
claims = fetch_nfip_claims()
housing = pd.read_csv(config.RAW / "housing_units.csv", dtype={"GEOID": str})
svi = pd.read_csv(config.RAW / "svi_florida.csv", dtype={"FIPS": str})
svi = svi.rename(columns={"FIPS": "GEOID", "RPL_THEMES": "svi"})[["GEOID", "svi"]]
svi["svi"] = svi["svi"].where(svi["svi"] >= 0)   # SVI uses -999 for missing

base = (t[["GEOID", "tract_area", "sfha_share", "geometry"]]
        .merge(claims, on="GEOID", how="left")
        .merge(housing, on="GEOID", how="left")
        .merge(svi, on="GEOID", how="left"))
base[["claims", "paid_total"]] = base[["claims", "paid_total"]].fillna(0)
base["claims_rate"] = base["claims"] / base["housing_units"].where(base["housing_units"] > 0) * 1000
base.isna().sum()

In [ ]:
base.to_crs(config.WEB_CRS).to_file(config.PROCESSED / "tracts_base.geojson", driver="GeoJSON")
print("saved", config.PROCESSED / "tracts_base.geojson")